# Milepost — from a number nobody chose to something people use

**Milepost** is a used-car marketplace. It has one model: given a car, estimate what it is
worth today. Somebody trained it, it works, and it is sitting on disk as a `.joblib` file.

Between that file and a product there are exactly two gaps, and this notebook is one gap each:

| Gap | Symptom | Part |
|---|---|---|
| **The model is a pile of defaults nobody chose** | Every hyperparameter is whatever the library shipped | P1 – P5, with **Optuna** |
| **Nobody but you can run it** | The only interface is a Python REPL with the right `import` | P6 – P7, with **Streamlit** and **Gradio** |

These are not two unrelated halves. The artifact the apps serve in P6 and P7 is the one the
tuning run produces in P5, and the app shows you which trial it came from. That is the whole
notebook in one sentence: **tune a model until it is worth shipping, then ship it to a human.**

## The map

```
                 MILEPOST — from a guess to something people use

  data/                      P1                     P2 · P3 · P4 · P5
  ─────                      ──                     ─────────────────
  cars24-car-price.csv ─► baseline XGBRegressor ─► an Optuna study ─► artifacts/
                          (defaults nobody chose)  (search, prune)    price_model.joblib
                                  │                                   baseline_model.joblib
                                  │                                   study.db
                                  │                                   model_card.json
      ┌───────────────────────────┴──────────────────────────────────────────┘
      │      surface           file             consumer                 what it cannot do
      │      ───────           ────             ────────                 ─────────────────
      ├─ P6  STREAMLIT   price_app.py     an analyst on your team   be called by a program
      └─ P7  GRADIO      gradio_app.py    someone you send a link   grow past a demo

      P1  can your metric tell two models apart?      P8  what the next class picks up
```

Parts 1, 5, 6 and 7 each end with a file you can run. Parts 2, 3 and 4 are the tuning itself.

In [1]:
%pip install -q optuna streamlit gradio scikit-learn xgboost torch plotly joblib pandas numpy ipython-autotime


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
%load_ext autotime

time: 110 µs (started: 2026-09-21 19:28:19 +05:30)


In [3]:
import json
import time
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import plotly.io as pio

# Optuna's plots are plotly figures. Pinning the renderer keeps them interactive in
# VS Code and Jupyter without embedding a 4.5 MB copy of plotly.js in this file.
pio.renderers.default = "plotly_mimetype+notebook_connected"

DATA = Path("data")
ARTIFACTS = Path("artifacts")
ARTIFACTS.mkdir(exist_ok=True)

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 30)
pd.set_option("future.no_silent_downcasting", True)
warnings.filterwarnings("ignore", category=FutureWarning)

import optuna as _optuna, sklearn as _sklearn, xgboost as _xgboost, torch as _torch

# Defaults, APIs and plot styles all move between releases. Pin what the lecture ran on.
print("versions        :", f"optuna {_optuna.__version__} · scikit-learn {_sklearn.__version__} "
      f"· xgboost {_xgboost.__version__} · torch {_torch.__version__}")
print("data files      :", sorted(p.name for p in DATA.glob("*.csv")))
print("artifacts so far:", sorted(p.name for p in ARTIFACTS.glob("*")) or "(empty — Part 5 fills this)")

versions        : optuna 4.9.0 · scikit-learn 1.7.2 · xgboost 3.2.0 · torch 2.10.0
data files      : ['cars24-car-price.csv', 'ticker_history.csv']
artifacts so far: (empty — Part 5 fills this)
time: 2.53 s (started: 2026-09-21 19:28:19 +05:30)


```
P0 setup  [ P1 THE BASELINE ]  P2 optuna  P3 vs grid search  P4 pruning  P5 the artifact  P6 streamlit  P7 gradio  P8 ship
```

# Part 1 — The baseline, and a metric that cannot tell two models apart

Before tuning anything you need two things: a model that runs, and **a number that moves when
the model gets better and stays put when it does not.** The second one is the one people skip,
and skipping it makes every tuning run that follows a waste of electricity.

## 1.1 The data

19,980 used cars, nine features, one target: `selling_price`, in lakhs of rupees.

In [4]:
cars = pd.read_csv(DATA / "cars24-car-price.csv")
print(cars.shape)
cars.head()

(19980, 11)


,full_name,selling_price,year,seller_type,km_driven,fuel_type,transmission_type,mileage,engine,max_power,seats
0,Maruti Alto Std,1.20,2012.0,Individual,120000,Petrol,Manual,19.70,796.0,46.30,5.0
1,Hyundai Grand i10 Asta,5.50,2016.0,Individual,20000,Petrol,Manual,18.90,1197.0,82.00,5.0
2,Hyundai i20 Asta,2.15,2010.0,Individual,60000,Petrol,Manual,17.00,1197.0,80.00,5.0
3,Maruti Alto K10 2010-2014 VXI,2.26,2012.0,Individual,37000,Petrol,Manual,20.92,998.0,67.10,5.0
4,Ford Ecosport 2015-2021 1.5 TDCi Titanium BSIV,5.70,2015.0,Dealer,30000,Diesel,Manual,22.77,1498.0,98.59,5.0


time: 24.1 ms (started: 2026-09-21 19:28:22 +05:30)


In [5]:
cars["selling_price"].describe()

count    19980.000000
mean         7.392066
std          9.103088
min          0.250000
25%          3.400000
50%          5.200000
75%          7.850000
max        395.000000
Name: selling_price, dtype: float64

time: 2.44 ms (started: 2026-09-21 19:28:22 +05:30)


Read the last three rows of that summary before going any further.

The median car is **5.2 lakhs**. The most expensive one is **395**. Three quarters of the
dataset sits under 7.85, and then there is a long thin tail of supercars running two orders of
magnitude above it.

In [6]:
print(f"skew        : {cars['selling_price'].skew():.2f}")
print(f"median      : {cars['selling_price'].median():.2f} lakhs")
print(f"99th pct    : {cars['selling_price'].quantile(0.99):.2f} lakhs")
print(f"max         : {cars['selling_price'].max():.2f} lakhs")
print(f"cars over 50: {(cars['selling_price'] > 50).sum()} of {len(cars)}")

skew        : 9.27
median      : 5.20 lakhs
99th pct    : 45.00 lakhs
max         : 395.00 lakhs
cars over 50: 150 of 19980
time: 1.52 ms (started: 2026-09-21 19:28:22 +05:30)


A skew of 9.3 is not a detail. It decides which metric you are allowed to trust, which is
section 1.3 — but first, a model to measure.

## 1.2 The baseline model

Three categorical columns get mapped to integers. The same dictionary has to be used at
prediction time — if it drifts, the model keeps answering, just wrongly.

In [7]:
encode_dict = {
    "fuel_type": {"Diesel": 1, "Petrol": 2, "CNG": 3, "LPG": 4, "Electric": 5},
    "transmission_type": {"Manual": 1, "Automatic": 2},
    "seller_type": {"Dealer": 1, "Individual": 2, "Trustmark Dealer": 3},
}

FEATURES = [
    "year",
    "seller_type",
    "km_driven",
    "fuel_type",
    "transmission_type",
    "mileage",
    "engine",
    "max_power",
    "seats",
]

df = cars.drop(columns=["full_name"]).replace(encode_dict).infer_objects(copy=False)
X, y = df[FEATURES], df["selling_price"]
X.head()

,year,seller_type,km_driven,fuel_type,transmission_type,mileage,engine,max_power,seats
0,2012.0,2,120000,2,1,19.70,796.0,46.30,5.0
1,2016.0,2,20000,2,1,18.90,1197.0,82.00,5.0
2,2010.0,2,60000,2,1,17.00,1197.0,80.00,5.0
3,2012.0,2,37000,2,1,20.92,998.0,67.10,5.0
4,2015.0,1,30000,1,1,22.77,1498.0,98.59,5.0


time: 14.8 ms (started: 2026-09-21 19:28:22 +05:30)


In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("train:", X_train.shape, " test:", X_test.shape)

train: (15984, 9)  test: (3996, 9)
time: 1.72 ms (started: 2026-09-21 19:28:22 +05:30)


In [9]:
from sklearn.metrics import mean_absolute_error, r2_score
from xgboost import XGBRegressor

baseline = XGBRegressor(random_state=42).fit(X_train, y_train)
pred = baseline.predict(X_test)

print(f"test R^2 : {r2_score(y_test, pred):.4f}")
print(f"test MAE : {mean_absolute_error(y_test, pred):.4f} lakhs")

test R^2 : 0.8749
test MAE : 1.0244 lakhs
time: 245 ms (started: 2026-09-21 19:28:22 +05:30)


An R² of 0.87 looks like a finished job. Here is the entire argument of this Part: **that number
is not a property of the model.**

## 1.3 Change nothing, and watch the score move

Same model. Same hyperparameters. Same data. The *only* thing that changes below is
`random_state` in the train/test split — which car lands in which half.

In [10]:
rows = []
for seed in range(6):
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=seed)
    m = XGBRegressor(random_state=42).fit(Xtr, ytr)
    p = m.predict(Xte)
    rows.append(
        {
            "split seed": seed,
            "R^2": r2_score(yte, p),
            "MAE": mean_absolute_error(yte, p),
            "test variance": yte.var(),
            "priciest car in test": yte.max(),
        }
    )

stability = pd.DataFrame(rows).set_index("split seed")
print(stability.round(3).to_string())
print()
print(f"R^2 : spread {stability['R^2'].max() - stability['R^2'].min():.4f}"
      f"   relative spread {stability['R^2'].std() / stability['R^2'].mean() * 100:.1f}%")
print(f"MAE : spread {stability['MAE'].max() - stability['MAE'].min():.4f}"
      f"   relative spread {stability['MAE'].std() / stability['MAE'].mean() * 100:.1f}%")
print()
print(f"corr(R^2, test-set variance) : {stability['R^2'].corr(stability['test variance']):+.3f}")

              R^2    MAE  test variance  priciest car in test
split seed                                                   
0           0.930  0.979         70.236                 132.0
1           0.812  1.056         69.068                 145.0
2           0.890  1.073         63.618                 111.0
3           0.902  1.031         64.834                 111.0
4           0.925  0.979         69.268                 111.0
5           0.752  1.123        102.455                 395.0

R^2 : spread 0.1775   relative spread 8.2%
MAE : spread 0.1438   relative spread 5.4%

corr(R^2, test-set variance) : -0.783
time: 1.5 s (started: 2026-09-21 19:28:22 +05:30)


**R² swings about 0.18 for a model that did not change.** Look at the last two columns to see
why — and in particular at seed 5.

Seed 5 is the split that happened to put the ₹395 lakh car in the test set. Its test variance
jumps from around 65 to over 100, and R² collapses from ~0.90 to 0.75. **One row out of 3,996
moved the headline score by fifteen points.**

That is not bad luck, it is what R² is. R² is *"what fraction of this test set's variance did I
explain"* — and the denominator is the test set's own variance, which the table shows changing
by 60% from split to split. The correlation between R² and test-set variance is about −0.8. So
0.93 and 0.75 are not two measurements of the same quantity that disagree; they are
measurements of two different quantities, and neither of them is "how good is the model".

MAE has no denominator. It is in lakhs, it means the same thing in every split, and it is the
number the business can act on: *we are off by about a lakh.* It still moves with the split —
5.4% against R²'s 8.2% — but it moves for an honest reason, and it never stops being comparable.

Two rules come out of this, and the rest of the notebook runs on both:

> **1. Optimise a metric in the units of the problem, not a normalised one.** → MAE, minimised.
>
> **2. Score every candidate on identical folds,** so the split cannot vary between them at all.
> → a fixed 3-fold cross-validation on the training set.

The test set is not touched again until Part 5.

In [11]:
from sklearn.model_selection import KFold, cross_val_score

CV = KFold(n_splits=3, shuffle=True, random_state=0)


def cv_mae(model):
    """Mean absolute error across 3 folds of the training set. Lower is better."""
    scores = cross_val_score(
        model, X_train, y_train, cv=CV, scoring="neg_mean_absolute_error"
    )
    return -scores.mean()


baseline_cv = cv_mae(XGBRegressor(random_state=42))
baseline_test = mean_absolute_error(y_test, baseline.predict(X_test))

print(f"baseline  cv MAE : {baseline_cv:.4f}")
print(f"baseline test MAE: {baseline_test:.4f}   <- not looked at again until Part 5")

baseline  cv MAE : 1.1470
baseline test MAE: 1.0244   <- not looked at again until Part 5
time: 721 ms (started: 2026-09-21 19:28:23 +05:30)


## 1.4 Nobody chose these numbers

That baseline was built with `XGBRegressor(random_state=42)`. Every other setting is whatever
the library ships.

In [12]:
defaults = XGBRegressor()
shipped = {
    k: getattr(defaults, k)
    for k in ["n_estimators", "max_depth", "learning_rate", "subsample", "colsample_bytree", "reg_lambda"]
}
for k, v in shipped.items():
    print(f"  {k:18s} {v}")

  n_estimators       None
  max_depth          None
  learning_rate      None
  subsample          None
  colsample_bytree   None
  reg_lambda         None
time: 362 µs (started: 2026-09-21 19:28:24 +05:30)


`None` means "whatever the C++ library decides at fit time". These are not recommendations for
your data — nobody at XGBoost has seen your data. They are the values that were least likely to
embarrass the library across every dataset in the world at once.

A **parameter** is something the model learns from data: the split thresholds inside each tree.
A **hyperparameter** is something you hand the model *before* it learns anything: how many
trees, how deep, how fast. The model cannot learn those, because they are the rules of learning
itself.

Which leaves the question the rest of the tuning half answers: given a budget of a few hundred
model fits, **how do you spend it?**

```
P0 setup  P1 the baseline  [ P2 OPTUNA ]  P3 vs grid search  P4 pruning  P5 the artifact  P6 streamlit  P7 gradio  P8 ship
```

# Part 2 — Optuna: the search space is code

Everyone writes the same first version of hyperparameter tuning, and it is a nested loop.

## 2.1 The loop everyone writes first

```python
best = None
for n in [200, 400, 800]:
    for d in [4, 6, 8]:
        for lr in [0.03, 0.1, 0.3]:
            score = cv_mae(XGBRegressor(n_estimators=n, max_depth=d, learning_rate=lr))
            if best is None or score < best[0]:
                best = (score, n, d, lr)
```

It works. Three things break it, in this order:

1. **It multiplies.** Three values for three parameters is 27 fits. Add a fourth parameter and
   it is 81. A fifth, 243. You did not make the search smarter, you made it longer.
2. **Everything has to be a list.** `learning_rate` is a positive real number. You are pretending
   it has three legal values because a `for` loop needs something to iterate over.
3. **It learns nothing.** Combination 27 is chosen with exactly as much information as
   combination 1, even though 26 results are sitting right there.

Optuna is the same loop with those three fixed.

## 2.2 The objective function

An Optuna objective is a plain Python function. It takes a `trial` and returns **one number**
— the thing to minimise or maximise. Inside, `trial.suggest_*` asks for a value, and Optuna
decides what to hand back.

In [13]:
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)  # otherwise every trial prints a line


def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 600),
        "max_depth": trial.suggest_int("max_depth", 3, 8),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
    }
    return cv_mae(XGBRegressor(random_state=42, **params))

time: 380 µs (started: 2026-09-21 19:28:24 +05:30)


Read what those six lines actually say, because the differences from the nested loop are the
entire point:

| | Nested loop | `trial.suggest_*` |
|---|---|---|
| `n_estimators` | three values you picked | **any integer** in 200–600 |
| `learning_rate` | three values you picked | **any real** in 0.01–0.3 |
| Cost of a 4th parameter | ×3 the combinations | **no extra trials** — though a bigger space may need more of them |
| Who picks the next combination | your `for` loop, in order | Optuna, from what it has seen |

`log=True` on `learning_rate` and `reg_lambda` is doing real work. Those parameters matter on a
*multiplicative* scale: the gap between 0.01 and 0.02 is the same kind of change as the gap
between 0.1 and 0.2. Sampling uniformly from 0.01–0.3 would put 2/3 of your trials above 0.1.
`log=True` samples the exponent instead, so every order of magnitude gets equal attention.

The suggest methods you will actually use:

| Call | Gives you |
|---|---|
| `trial.suggest_int("n", lo, hi)` | an integer, optionally `step=` or `log=True` |
| `trial.suggest_float("lr", lo, hi)` | a float, optionally `step=` or `log=True` |
| `trial.suggest_categorical("kind", ["a", "b"])` | one of a fixed list — for things with no order |

## 2.3 The study

A **study** is the search. You give it a direction, hand it the objective, and say how many
trials to spend.

In [14]:
study = optuna.create_study(
    direction="minimize",                              # MAE: lower is better
    sampler=optuna.samplers.TPESampler(seed=7),        # seeded so this notebook reproduces
    study_name="milepost-price-demo",
)

study.optimize(objective, n_trials=12)

print(f"trials run : {len(study.trials)}")
print(f"best cv MAE: {study.best_value:.4f}   (baseline was {baseline_cv:.4f})")
print("best params:")
for k, v in study.best_params.items():
    print(f"  {k:18s} {v:.4f}" if isinstance(v, float) else f"  {k:18s} {v}")

trials run : 12
best cv MAE: 1.1176   (baseline was 1.1470)
best params:
  n_estimators       212
  max_depth          7
  learning_rate      0.1317
  subsample          0.7831
  colsample_bytree   0.9909
  reg_lambda         0.0073
time: 25.3 s (started: 2026-09-21 19:28:24 +05:30)


Twelve trials, and `study.best_params` is a dictionary you can splat straight into a model.
Every trial is kept, not just the winner:

In [15]:
trials = study.trials_dataframe()[
    ["number", "value", "params_max_depth", "params_learning_rate", "duration"]
]

# TPESampler draws at random until it has n_startup_trials finished results to model from.
n_startup = getattr(study.sampler, "_n_startup_trials", 10)
trials["sampled by"] = ["random (startup)" if n < n_startup else "TPE model" for n in trials["number"]]
trials

,number,value,params_max_depth,params_learning_rate,duration,sampled by
0,0,1.145706,7,0.044421,0 days 00:00:01.957659,random (startup)
1,1,1.474995,3,0.024918,0 days 00:00:01.626731,random (startup)
2,2,1.435396,3,0.026646,0 days 00:00:01.438079,random (startup)
3,3,1.255382,3,0.077105,0 days 00:00:02.300985,random (startup)
4,4,1.281329,3,0.059312,0 days 00:00:02.310536,random (startup)
5,5,1.288167,5,0.035486,0 days 00:00:01.748667,random (startup)
6,6,1.189936,4,0.070119,0 days 00:00:02.581704,random (startup)
7,7,1.213836,5,0.047658,0 days 00:00:02.851068,random (startup)
8,8,1.134934,7,0.042063,0 days 00:00:02.336688,random (startup)
9,9,1.560489,3,0.013686,0 days 00:00:01.486809,random (startup)


time: 4.54 ms (started: 2026-09-21 19:28:49 +05:30)


### Most of that study was not TPE

Look at the last column before going further. `TPESampler` takes an `n_startup_trials`
argument that defaults to **10**: it cannot model which regions of the space are good until it
has results to model, so the first ten trials are drawn **at random**. Only the last two above
were chosen by TPE.

So a 12-trial study is a 10-trial random search with two informed guesses stapled on, and a
10-trial study is random search with extra steps. That is worth knowing before running
something small, seeing no magic, and concluding Optuna does not help — at that size, it
genuinely is not doing the thing it is famous for.

None of that spoils this section, because the point here was the *API*: an objective that
returns one number, a study that collects trials, and `best_params` at the end. But it does
set up Part 3, which runs to 40 trials for exactly this reason — and which finds that even at
27, TPE has barely got going.

## 2.4 Where define-by-run becomes much more natural

Everything so far could have been a `RandomizedSearchCV`. This cannot:

```python
def objective(trial):
    kind = trial.suggest_categorical("model", ["xgb", "ridge"])
    if kind == "xgb":
        params = {"max_depth": trial.suggest_int("max_depth", 3, 10)}   # only exists for xgb
    else:
        params = {"alpha": trial.suggest_float("alpha", 1e-3, 10, log=True)}
    ...
```

The search space is **built while the trial runs**, so it can contain `if`. Optuna calls this
*define-by-run*, and it is why `max_depth` can exist only on the branches where it means
something.

To be fair to `GridSearchCV`: it is **not** true that a grid cannot express this. Hand it a
*list* of dictionaries and each one becomes its own subspace, with its own keys:

```python
GridSearchCV(est, [
    {"model": ["xgb"],   "max_depth": [3, 6, 10]},
    {"model": ["ridge"], "alpha": [1e-3, 1e-1, 10]},
])
```

That works, and `RandomizedSearchCV` takes the same form. The difference is not capability,
it is what you have to write. Every branch is enumerated by hand, every value stays discrete,
and the list grows with the product of the options inside each subspace.

Part 4 is where that stops being a style preference: a neural network whose *number of layers*
is a hyperparameter, so `units_l0`, `units_l1`, `units_l2` come into existence one per layer.
As a list of dictionaries that is one subspace per depth, each holding the full cross-product
of its own layer widths. As define-by-run it is a `for` loop.

```
P0 setup  P1 the baseline  P2 optuna  [ P3 VS GRID SEARCH ]  P4 pruning  P5 the artifact  P6 streamlit  P7 gradio  P8 ship
```

# Part 3 — Against grid search, measured

Nothing below is asserted; it is run in front of you, and the numbers are whatever they are.

There are **two different questions** here, and they are easy to run together and then draw
the wrong conclusion from. Keeping them apart is the whole design of this Part:

| | Question | What changes | What stays fixed |
|---|---|---|---|
| **A** | Does a richer, continuous search space beat a hand-written grid? | the space (3 discrete params → 6 continuous) *and* the algorithm | metric, data, folds, ~candidate count |
| **B** | Does adaptive sampling beat random sampling? | **only the sampler** | metric, data, folds, objective, search space, trial count |

**B is the clean experiment.** `RandomSampler` and `TPESampler` run the identical objective
over the identical space for the identical number of trials, so any difference is the sampler
and nothing else.

**A is deliberately confounded** — two things change at once — and it has to be, because the
comparison people actually face is *"the grid I would have written"* against *"the Optuna
objective I would have written"*, and nobody writes a six-dimensional continuous grid. So read
A as a comparison of two **workflows**, not of two algorithms, and let B carry the claim about
sampling.

One thing that is *not* held fixed anywhere: wall clock. The searches get a comparable number
of candidates, not a comparable number of seconds, and §3.3 shows how far apart those two
things drift.

## 3.1 Grid search, doing its best

To keep this fair, the grid gets the three hyperparameters that matter most and three
sensible values each — the grid a careful person would actually write. That is 27 combinations
× 3 folds = **81 model fits**.

In [16]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "n_estimators": [200, 400, 600],
    "max_depth": [4, 6, 8],
    "learning_rate": [0.03, 0.1, 0.3],
}

t0 = time.perf_counter()
grid = GridSearchCV(
    XGBRegressor(random_state=42),
    param_grid,
    cv=CV,
    scoring="neg_mean_absolute_error",
    n_jobs=1,
).fit(X_train, y_train)
grid_seconds = time.perf_counter() - t0

print(f"combinations : {len(grid.cv_results_['params'])}")
print(f"model fits   : {len(grid.cv_results_['params']) * CV.get_n_splits()}")
print(f"wall clock   : {grid_seconds:.0f}s")
print(f"best cv MAE  : {-grid.best_score_:.4f}")
print(f"best params  : {grid.best_params_}")

combinations : 27
model fits   : 81
wall clock   : 79s
best cv MAE  : 1.0953
best params  : {'learning_rate': 0.1, 'max_depth': 6, 'n_estimators': 600}
time: 1min 18s (started: 2026-09-21 19:28:49 +05:30)


### Why the grid stops at three parameters

It is not laziness. It is multiplication.

In [17]:
per_fit = grid_seconds / (len(grid.cv_results_["params"]) * CV.get_n_splits())

print(f"one fit ≈ {per_fit:.2f}s\n")
print(f"{'parameters':>11}  {'combinations':>13}  {'fits':>7}  {'wall clock':>12}")
for k in range(2, 8):
    combos = 3 ** k
    secs = combos * CV.get_n_splits() * per_fit
    pretty = f"{secs:.0f}s" if secs < 120 else (f"{secs/60:.0f} min" if secs < 7200 else f"{secs/3600:.1f} hours")
    print(f"{k:>11}  {combos:>13}  {combos * CV.get_n_splits():>7}  {pretty:>12}")

one fit ≈ 0.97s

 parameters   combinations     fits    wall clock
          2              9       27           26s
          3             27       81           79s
          4             81      243         4 min
          5            243      729        12 min
          6            729     2187        35 min
          7           2187     6561       106 min
time: 525 µs (started: 2026-09-21 19:30:08 +05:30)


Three values per parameter is already a coarse grid — and the cost of adding the *fourth*
parameter is the whole search over again, three times. This is why grid searches in the wild
have three or four parameters and why the values are always round numbers. The method forces it.

Optuna's cost of a fourth parameter is zero: the trial count is whatever you set, no matter how
many dimensions it is searching.

## 3.2 Optuna, same candidate budget  —  questions A and B

Two studies over the **six**-parameter continuous space from Part 2, run to 40 trials each.
The samplers are the only difference:

- **`RandomSampler`** — draw each parameter at random from its range. Learns nothing.
- **`TPESampler`** — Optuna's default. After the first 10 trials it fits a density model over
  which regions produced good scores and samples from there.

`RandomSampler` is the control that makes **question B** answerable. Both studies call the same
`objective`, over the same six-dimensional space, for the same 40 trials. The only difference
in the two lines below is the sampler object — so whatever separates them is attributable to
adaptive sampling and to nothing else.

In [18]:
N_TRIALS = 40
studies = {}

for name, sampler in [
    ("random", optuna.samplers.RandomSampler(seed=7)),
    ("TPE", optuna.samplers.TPESampler(seed=7)),
]:
    t0 = time.perf_counter()
    s = optuna.create_study(direction="minimize", sampler=sampler, study_name=f"price-{name}")
    s.optimize(objective, n_trials=N_TRIALS)
    studies[name] = {"study": s, "seconds": time.perf_counter() - t0}
    print(f"{name:7s} {N_TRIALS} trials  best cv MAE {s.best_value:.4f}  in {studies[name]['seconds']:.0f}s")

random  40 trials  best cv MAE 1.0956  in 117s


TPE     40 trials  best cv MAE 1.0872  in 131s
time: 4min 7s (started: 2026-09-21 19:30:08 +05:30)


## 3.3 The comparison

Grid search got 27 combinations, so it is compared against the **first 27 trials** of each
study as well as the full 40. Every study keeps its whole history, so reading a prefix costs
nothing.

In [ ]:
def best_so_far(study, upto):
    return min(t.value for t in study.trials[:upto] if t.value is not None)


def wall_for(study, upto):
    """Real seconds spent on the first `upto` trials, summed from their own timings."""
    return sum(t.duration.total_seconds() for t in study.trials[:upto])


rows = [
    {"search": "baseline (no tuning)", "params searched": 0, "trials": 0, "cv MAE": baseline_cv, "wall": 0.0},
    {"search": "GridSearchCV", "params searched": 3, "trials": 27, "cv MAE": -grid.best_score_, "wall": grid_seconds},
]


for trials in (27, N_TRIALS):
    for name in ("random", "TPE"):
        s = studies[name]["study"]
        rows.append({"search": f"Optuna {name}", "params searched": 6, "trials": trials,
                     "cv MAE": best_so_far(s, trials), "wall": wall_for(s, trials)})

comparison = pd.DataFrame(rows)
comparison["vs baseline"] = (baseline_cv - comparison["cv MAE"]).round(4)
print(comparison.round({"cv MAE": 4, "wall": 0}).to_string(index=False))

              search  params searched  trials  cv MAE  wall  vs baseline
baseline (no tuning)                0       0  1.1470   0.0       0.0000
        GridSearchCV                3      27  1.0953  79.0       0.0518
       Optuna random                6      27  1.0956  79.0       0.0514
          Optuna TPE                6      27  1.0995  73.0       0.0475
       Optuna random                6      40  1.0956 117.0       0.0514
          Optuna TPE                6      40  1.0872 131.0       0.0598
time: 14.8 ms (started: 2026-09-21 19:34:16 +05:30)


In [20]:
# Where the trials actually landed -- the best score is one number, the distribution is the story.
summary = pd.DataFrame(
    {
        name: {
            "best": min(t.value for t in d["study"].trials),
            "median trial": float(np.median([t.value for t in d["study"].trials])),
            "median of last 20": float(np.median([t.value for t in d["study"].trials[-20:]])),
            "trials under 1.10": int(sum(t.value < 1.10 for t in d["study"].trials)),
        }
        for name, d in studies.items()
    }
).T.round(4)
print(summary.to_string())

          best  median trial  median of last 20  trials under 1.10
random  1.0956        1.1767             1.1622                1.0
TPE     1.0872        1.1288             1.1135                6.0
time: 13.5 ms (started: 2026-09-21 19:34:16 +05:30)


### A trial budget is not a time budget

Trials are not all the same price. More trees and deeper ones cost more to fit — and on this
data they also tend to *score* better. So a sampler that converges on the good region is also
converging on the expensive region, and its trials get slower as it succeeds.

Every trial's duration is recorded, so this is checkable rather than a story:

In [21]:
for name, d in studies.items():
    dur = [t.duration.total_seconds() for t in d["study"].trials]
    first, last = np.mean(dur[:10]), np.mean(dur[-10:])
    print(f"{name:7s}  first 10 trials {first:5.2f}s/trial   "
          f"last 10 {last:5.2f}s/trial   ratio {last / first:4.1f}x")

random   first 10 trials  2.09s/trial   last 10  3.08s/trial   ratio  1.5x
TPE      first 10 trials  2.15s/trial   last 10  4.65s/trial   ratio  2.2x
time: 3.3 ms (started: 2026-09-21 19:34:16 +05:30)


Worth knowing before promising anyone that "500 trials" will be done by lunchtime. If a search
has to finish inside a fixed window, the lever is not the trial count — it is the range of the
*expensive* hyperparameters. `n_estimators` capped at 600 rather than 2000 sets the price of
every trial in the search, including the ones you have not run yet.

## 3.4 Seeing it

`plot_optimization_history` draws every trial as a dot and the running best as a line. Run it
for both samplers and the difference in *behaviour* is visible even where the best values are
close: random keeps scattering across the whole space forever, TPE's cloud collapses downward
once it has something to go on.

In [22]:
from optuna.visualization import (
    plot_optimization_history,
    plot_param_importances,
    plot_slice,
)

plot_optimization_history(studies["random"]["study"]).update_layout(
    title="RandomSampler — 40 trials", height=380
).show()
plot_optimization_history(studies["TPE"]["study"]).update_layout(
    title="TPESampler — 40 trials", height=380
).show()

time: 232 ms (started: 2026-09-21 19:34:16 +05:30)


### Which knobs mattered

The other reason to keep every trial: with 40 results across six dimensions, Optuna can
estimate which hyperparameters were most associated with a good score **within the region this
study actually explored**. That is a weaker claim than "which parameters matter", and the
difference is worth holding on to — the estimate is computed from 40 trials that the sampler
deliberately clustered in one part of the space, not from a sweep of the whole thing. Read it
as exploratory.

In [23]:
plot_param_importances(studies["TPE"]["study"]).update_layout(height=380).show()

time: 462 ms (started: 2026-09-21 19:34:16 +05:30)


In [24]:
plot_slice(studies["TPE"]["study"], params=["max_depth", "learning_rate", "n_estimators"]).update_layout(
    height=380
).show()

time: 32.6 ms (started: 2026-09-21 19:34:16 +05:30)


Read the slice plot as *"for this one parameter, what did the score do?"* — a visible trough
suggests a promising range, a flat smear suggests the score was not very sensitive to it here.
Both readings are provisional: each panel flattens five other dimensions onto one axis, so an
interaction between two parameters can hide completely. Use it to pick where to look next, not
to decide what matters.

None of this is unique to Optuna — `GridSearchCV` keeps every configuration it tried in
`cv_results_`, and you can plot that too. What Optuna gives you is that the history, the
plots, the adaptive sampling, the pruning in Part 4 and the persistence in Part 5 are all the
same object, so none of it has to be assembled by hand.

```
P0 setup  P1 the baseline  P2 optuna  P3 vs grid search  [ P4 PRUNING ]  P5 the artifact  P6 streamlit  P7 gradio  P8 ship
```

# Part 4 — Pruning: stop training the ones that are already losing

Look again at the trial distribution from Part 3. Most trials are not close. The search spends
almost all of its time training models that were never going to win — and with XGBoost at a
second a fit, that is annoying but survivable.

Now make each fit take four minutes, which is what happens the moment a neural network is
involved. A 60-trial search becomes four hours, and three and a half of those hours are spent
finishing models that were visibly bad after thirty seconds.

**Pruning** is the fix: report your score as training goes, and let Optuna kill the trial when
it is clearly behind.

## 4.1 A model that trains in visible steps

Pruning needs a model with *intermediate* scores — one that gets better in observable steps
rather than returning a single number at the end. A neural network trained over epochs is the
clearest case, so here is a small MLP on the same Milepost pricing problem.

`nn.L1Loss` is mean absolute error, which keeps the metric identical to everything above — and
which matters twice as much here, because a squared loss on a target with skew 9.3 would spend
the entire training budget on supercars.

In [25]:
import torch
import torch.nn as nn

torch.manual_seed(0)

# These batches are 256x9. Spread across every core, the thread synchronisation
# costs more than the arithmetic -- one thread is measurably faster here.
torch.set_num_threads(1)

from sklearn.preprocessing import StandardScaler

# A validation split carved out of the TRAINING data. The test set is still untouched.
X_fit, X_val, y_fit, y_val = train_test_split(X_train, y_train.values, test_size=0.2, random_state=1)

# Trees split on thresholds, so feature scale is irrelevant to them -- nothing above this
# point was scaled. A network multiplies its inputs by weights, so scale matters a great
# deal here. Note where the scaler is fitted: on X_fit ONLY. Fitting it on all of X_train
# would let the validation rows influence the numbers the model is then judged against.
mlp_scaler = StandardScaler().fit(X_fit)

t_X_fit = torch.tensor(mlp_scaler.transform(X_fit), dtype=torch.float32)
t_y_fit = torch.tensor(y_fit, dtype=torch.float32).unsqueeze(1)
t_X_val = torch.tensor(mlp_scaler.transform(X_val), dtype=torch.float32)
t_y_val = torch.tensor(y_val, dtype=torch.float32).unsqueeze(1)

print("fit:", tuple(t_X_fit.shape), " val:", tuple(t_X_val.shape))

fit: (12787, 9)  val: (3197, 9)
time: 5.42 ms (started: 2026-09-21 19:34:16 +05:30)


## 4.2 A search space whose *shape* changes

This is the define-by-run idea from §2.4, doing something a grid physically cannot.

`n_layers` is a hyperparameter. The loop below then asks for `units_l0`, `units_l1`, ... — one
per layer. So a 1-layer trial has 3 hyperparameters and a 3-layer trial has 7, and **which
hyperparameters exist depends on the value of another hyperparameter.** There is no dictionary
you can hand `GridSearchCV` that means this.

In [26]:
def build_model(trial):
    layers = []
    n_in = t_X_fit.shape[1]

    n_layers = trial.suggest_int("n_layers", 1, 3)
    for i in range(n_layers):
        # These parameter names only exist on trials that got this far in the loop.
        n_out = trial.suggest_int(f"units_l{i}", 16, 128, log=True)
        dropout = trial.suggest_float(f"dropout_l{i}", 0.0, 0.4)
        layers += [nn.Linear(n_in, n_out), nn.ReLU(), nn.Dropout(dropout)]
        n_in = n_out

    layers.append(nn.Linear(n_in, 1))
    return nn.Sequential(*layers)

time: 310 µs (started: 2026-09-21 19:34:16 +05:30)


## 4.3 The two lines that make pruning work

Everything about pruning is these two lines, at the bottom of the epoch loop:

```python
trial.report(val_mae, epoch)          # "here is how I am doing at step N"
if trial.should_prune():              # "...should I stop?"
    raise optuna.TrialPruned()
```

Two different jobs, and a study has one of each:

| | Decides | Runs |
|---|---|---|
| **Sampler** | *which configuration to try next* | once, before each trial starts |
| **Pruner** | *whether the trial now running is worth finishing* | at every step you report |

Part 3 changed the sampler and left the pruner alone. This section does the opposite.

`report` hands Optuna an intermediate value. `should_prune()` asks the study's **pruner**
whether this trial is worth continuing. The default, `MedianPruner`, answers *yes, stop* when
this trial's best score so far is worse than the median of what previous trials had reached by
the same epoch. Raising `TrialPruned` ends the trial — it is recorded as pruned rather than
failed, and the sampler still learns from the epochs it did see.

In [27]:
EPOCHS = 40
BATCH = 256


def mlp_objective(trial):
    model = build_model(trial)
    lr = trial.suggest_float("lr", 1e-4, 1e-1, log=True)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.L1Loss()
    n = len(t_X_fit)
    best_val = float("inf")

    for epoch in range(EPOCHS):
        model.train()
        order = torch.randperm(n)
        for i in range(0, n, BATCH):
            idx = order[i : i + BATCH]
            optimizer.zero_grad()
            loss_fn(model(t_X_fit[idx]), t_y_fit[idx]).backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_mae = loss_fn(model(t_X_val), t_y_val).item()
        best_val = min(best_val, val_mae)

        trial.report(val_mae, epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()

    return best_val

time: 453 µs (started: 2026-09-21 19:34:16 +05:30)


## 4.4 The same search, with and without the pruner

Identical sampler, identical seed, identical 25 trials. The only difference is the pruner, so
any difference in wall clock is the pruner's doing.

`NopPruner` never prunes — it is the control.

In [28]:
from optuna.trial import TrialState

mlp = {}
for name, pruner in [
    ("no pruner", optuna.pruners.NopPruner()),
    ("MedianPruner", optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=5)),
]:
    t0 = time.perf_counter()
    s = optuna.create_study(
        direction="minimize",
        sampler=optuna.samplers.TPESampler(seed=7),
        pruner=pruner,
        study_name=f"mlp-{name}",
    )
    s.optimize(mlp_objective, n_trials=25)
    elapsed = time.perf_counter() - t0
    pruned = [t for t in s.trials if t.state == TrialState.PRUNED]
    mlp[name] = {
        "study": s,
        "seconds": elapsed,
        "pruned": len(pruned),
        "epochs trained": sum(len(t.intermediate_values) for t in s.trials),
        "best val MAE": s.best_value,
    }
    print(
        f"{name:13s} {elapsed:6.0f}s   pruned {len(pruned):2d}/25   "
        f"epochs {mlp[name]['epochs trained']:4d}/{25 * EPOCHS}   best {s.best_value:.4f}"
    )

no pruner         19s   pruned  0/25   epochs 1000/1000   best 1.2587


MedianPruner      16s   pruned  9/25   epochs  720/1000   best 1.2610
time: 35.4 s (started: 2026-09-21 19:34:16 +05:30)


In [29]:
saving = pd.DataFrame(
    {k: {kk: vv for kk, vv in v.items() if kk != "study"} for k, v in mlp.items()}
).T
print(saving.round(4).to_string())

off, on = mlp["no pruner"], mlp["MedianPruner"]
print()
print(f"wall clock saved : {(1 - on['seconds'] / off['seconds']) * 100:5.1f}%"
      f"   ({off['seconds']:.0f}s -> {on['seconds']:.0f}s)")
print(f"epochs skipped   : {off['epochs trained'] - on['epochs trained']:5d}"
      f"   of {off['epochs trained']}")
print(f"cost in best MAE : {on['best val MAE'] - off['best val MAE']:+.4f}"
      f"   ({off['best val MAE']:.4f} -> {on['best val MAE']:.4f})")

              seconds  pruned  epochs trained  best val MAE
no pruner     19.3709     0.0          1000.0        1.2587
MedianPruner  15.9919     9.0           720.0        1.2610

wall clock saved :  17.4%   (19s -> 16s)
epochs skipped   :   280   of 1000
cost in best MAE : +0.0023   (1.2587 -> 1.2610)
time: 1.61 ms (started: 2026-09-21 19:34:52 +05:30)


Two numbers to read together, because either one alone is misleading:

- **epochs trained** is what the pruner actually did — the training that never happened.
- **best val MAE** is what it cost. Pruning is a bet that a trial behind at epoch 5 will still
  be behind at epoch 40. That bet is usually right and occasionally wrong — and the run above
  is one of the times it was wrong: the pruner killed a trial that would have come back, and
  the best score is slightly worse for it. That is the price, and it is normally worth paying.

`MedianPruner(n_startup_trials=5, n_warmup_steps=5)` is what keeps the bet sane: no pruning at
all until 5 trials have finished (there is no median to compare against yet), and no pruning
inside the first 5 epochs of any trial (everything looks bad at epoch 1).

In [30]:
from optuna.visualization import plot_intermediate_values

plot_intermediate_values(mlp["MedianPruner"]["study"]).update_layout(
    title="Each line is one trial's validation MAE per epoch — short lines were pruned",
    height=420,
).show()

time: 8.7 ms (started: 2026-09-21 19:34:52 +05:30)


That plot is pruning in one picture: the full-length lines are trials that were allowed to
finish, and every line that stops early is training you did not pay for.

Other pruners exist — `SuccessiveHalvingPruner` and `HyperbandPruner` are the ones worth
knowing, and they allocate budget more cleverly across trials rather than judging each one
against the median. `MedianPruner` is the default because it is nearly always good enough.

## 4.5 Did the neural network win?

The honest question, asked out loud, because a notebook that introduces a neural network and
then quietly never mentions it again has taught the wrong lesson.

First, a warning about how *not* to answer it. The obvious move is to print the MLP's best
validation MAE next to the XGBoost study's best cross-validation MAE — and those two numbers
are not comparable. They come from different procedures:

- the XGBoost number is a **mean over 3 CV folds**, each fold fitted from scratch;
- the MLP number is from **one fixed validation split**, taken as the best epoch, and that
  same split steered all 25 trials.

The second is optimistically biased in a way the first is not. To compare them, score both on
the same data with the same rule — so here is XGBoost fitted on `X_fit` and scored on `X_val`,
exactly as the network was.

In [31]:
xgb_same_split = XGBRegressor(random_state=42, **studies["TPE"]["study"].best_params).fit(X_fit, y_fit)
xgb_val_mae = mean_absolute_error(y_val, xgb_same_split.predict(X_val))

print("scored the same way -- fitted on X_fit, evaluated on X_val:")
print(f"  tuned XGBoost : {xgb_val_mae:.4f}")
print(f"  tuned MLP     : {mlp['MedianPruner']['best val MAE']:.4f}")
print()
print("for reference, on their own native protocols (NOT comparable to each other):")
print(f"  XGBoost, mean of 3 CV folds : {studies['TPE']['study'].best_value:.4f}")
print(f"  baseline XGBoost, 3 CV folds: {baseline_cv:.4f}")

scored the same way -- fitted on X_fit, evaluated on X_val:
  tuned XGBoost : 0.9691
  tuned MLP     : 1.2610

for reference, on their own native protocols (NOT comparable to each other):
  XGBoost, mean of 3 CV folds : 1.0872
  baseline XGBoost, 3 CV folds: 1.1470
time: 1.63 s (started: 2026-09-21 19:34:52 +05:30)


On tabular data of this size and shape, gradient-boosted trees are very hard to beat, and a
25-trial budget on an MLP was never going to do it. That is the expected result, not a
disappointment — and it is why the model that gets shipped in Part 5 is the XGBoost one.

What the MLP earned its place for is the *mechanism*: a search space whose shape changes per
trial, and a pruner that can watch training happen. Both of those transfer directly to the
model where they pay for themselves — the one that takes four minutes a fit.

```
P0 setup  P1 the baseline  P2 optuna  P3 vs grid search  P4 pruning  [ P5 THE ARTIFACT ]  P6 streamlit  P7 gradio  P8 ship
```

# Part 5 — The study is an artifact too

Everything in Parts 3 and 4 lives in the kernel. Restart it and several minutes of compute are
gone, along with every trial that explained *why* the winner won.

A tuning run is not a thing you scroll back to. It is a thing you keep.

## 5.1 A study that survives the kernel

One argument — `storage=` — and Optuna writes every trial to SQLite as it goes.
`load_if_exists=True` means re-running the cell continues the study instead of crashing on the
duplicate name.

The studies in Parts 3 and 4 were in memory, which is the right choice for a throwaway
experiment and the wrong one for the run that produces the model you ship.

In [32]:
STUDY_DB = f"sqlite:///{(ARTIFACTS / 'study.db').as_posix()}"
STUDY_NAME = "milepost-price"

persisted = optuna.create_study(
    study_name=STUDY_NAME,
    storage=STUDY_DB,
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=7),
    load_if_exists=True,
)

# Part 3 already found a good configuration. enqueue_trial puts it at the front of the
# queue so this study starts from it instead of rediscovering it from scratch.
persisted.enqueue_trial(studies["TPE"]["study"].best_params)

persisted.optimize(objective, n_trials=20)
print(f"trials in the database: {len(persisted.trials)}")
print(f"best cv MAE           : {persisted.best_value:.4f}")
print(f"file on disk          : {(ARTIFACTS / 'study.db').stat().st_size / 1024:.0f} KB")

trials in the database: 20
best cv MAE           : 1.0872
file on disk          : 128 KB
time: 1min 4s (started: 2026-09-21 19:34:53 +05:30)


Now prove it is really on disk. `load_study` reads it back by name — from a different process,
a different machine, or three weeks later:

In [33]:
reloaded = optuna.load_study(study_name=STUDY_NAME, storage=STUDY_DB)
print(f"reloaded {len(reloaded.trials)} trials, best {reloaded.best_value:.4f}")

# ...and it picks up where it stopped rather than starting over.
reloaded.optimize(objective, n_trials=10)
print(f"after 10 more         : {len(reloaded.trials)} trials, best {reloaded.best_value:.4f}")

reloaded 20 trials, best 1.0872


after 10 more         : 30 trials, best 1.0842
time: 45.5 s (started: 2026-09-21 19:35:58 +05:30)


That is the whole feature, and it is worth more than it looks:

- A search that takes six hours can be killed at hour three and resumed.
- The study is a **record**: six months later, "why is `max_depth` 9?" has an answer with 90
  trials behind it instead of a shrug.
- Several machines can run trials into the same database at once, because the storage is the
  coordination point.

## 5.2 Train the model you are actually going to ship

`best_params` came from cross-validation on the training set. The model that ships is refit on
**all** of the training data with those parameters — the CV was for choosing, not for the final
fit.

There is no preprocessing to carry along: nothing in the XGBoost path was ever scaled, because
a tree splits on thresholds and a monotonic rescaling cannot change where those thresholds
fall. The model *is* the artifact.

In [34]:
best_params = persisted.best_params

tuned_model = XGBRegressor(random_state=42, **best_params).fit(X_train, y_train)
baseline_model = XGBRegressor(random_state=42).fit(X_train, y_train)

tuned_test = mean_absolute_error(y_test, tuned_model.predict(X_test))
base_test = mean_absolute_error(y_test, baseline_model.predict(X_test))

print(f"baseline test MAE : {base_test:.4f} lakhs")
print(f"tuned    test MAE : {tuned_test:.4f} lakhs")
print(f"improvement       : {base_test - tuned_test:.4f} lakhs ({(base_test - tuned_test) / base_test * 100:.1f}%)")

baseline test MAE : 1.0244 lakhs
tuned    test MAE : 0.9687 lakhs
improvement       : 0.0557 lakhs (5.4%)
time: 1.94 s (started: 2026-09-21 19:36:44 +05:30)


### One split is not evidence

That improvement came from a single train/test split — and Part 1.3 is the reason to distrust
exactly that. The same model scored anywhere from 0.98 to 1.12 MAE depending only on which
cars landed in the test set, and the improvement just measured is 0.05.

So measure the *difference* the way Part 1 measured the metric: across several splits, with
baseline and tuned trained and scored on identical data each time.

In [35]:
deltas = []
for seed in range(8):
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=seed)
    b = XGBRegressor(random_state=42).fit(Xtr, ytr)
    t = XGBRegressor(random_state=42, **best_params).fit(Xtr, ytr)
    b_mae = mean_absolute_error(yte, b.predict(Xte))
    t_mae = mean_absolute_error(yte, t.predict(Xte))
    deltas.append({"seed": seed, "baseline": b_mae, "tuned": t_mae, "tuned - baseline": t_mae - b_mae})

robust = pd.DataFrame(deltas).set_index("seed")
print(robust.round(4).to_string())
print()
d = robust["tuned - baseline"]
print(f"mean improvement : {-d.mean():.4f} lakhs")
print(f"worst split      : {-d.max():+.4f}   best split: {-d.min():+.4f}")
print(f"splits improved  : {(d < 0).sum()} of {len(d)}")

      baseline   tuned  tuned - baseline
seed                                    
0       0.9791  0.9648           -0.0143
1       1.0556  1.0016           -0.0540
2       1.0733  1.0161           -0.0571
3       1.0309  0.9721           -0.0588
4       0.9788  0.9577           -0.0211
5       1.1226  1.0632           -0.0594
6       1.0345  1.0251           -0.0094
7       1.0988  1.0257           -0.0731

mean improvement : 0.0434 lakhs
worst split      : +0.0094   best split: +0.0731
splits improved  : 8 of 8
time: 15.3 s (started: 2026-09-21 19:36:46 +05:30)


This is the number to quote, and it is the honest end of the tuning half: not *"tuning gained
5%"* from one lucky split, but the improvement holding across splits the search never saw.
If it had held on only four of eight, the right conclusion would have been that 40 trials
bought nothing — and that is a conclusion worth being able to reach.

That is the first and only time the test set has been used, and it is the number Milepost gets
quoted — not the cross-validation score, which was the *selection* metric and is optimistically
biased by having been minimised directly.

## 5.3 What goes on disk

Two pipelines and a model card. The card is the thing that makes the artifact explainable six
months from now: it records which study produced this model, which trial, and what it scored.

In [36]:
joblib.dump(tuned_model, ARTIFACTS / "price_model.joblib")
joblib.dump(baseline_model, ARTIFACTS / "baseline_model.joblib")

model_card = {
    "model": "XGBRegressor (no preprocessing -- trees are scale-invariant)",
    "target": "selling_price (lakhs)",
    "features": FEATURES,
    "study_name": STUDY_NAME,
    "storage": "artifacts/study.db",
    "n_trials": len(persisted.trials),
    "best_trial": persisted.best_trial.number,
    "best_params": best_params,
    "cv_mae": round(persisted.best_value, 4),
    "test_mae": round(tuned_test, 4),
    "baseline_test_mae": round(base_test, 4),
    "test_mae_mean_over_8_splits": round(float(robust["tuned"].mean()), 4),
    "baseline_mae_mean_over_8_splits": round(float(robust["baseline"].mean()), 4),
    "metric_note": "MAE, not R^2 -- see Part 1.3; R^2 moves 0.18 on this target from the split seed alone",
}
(ARTIFACTS / "model_card.json").write_text(json.dumps(model_card, indent=2))

for p in sorted(ARTIFACTS.glob("*")):
    print(f"  {p.name:28s} {p.stat().st_size / 1024:8.1f} KB")

  baseline_model.joblib           392.9 KB
  model_card.json                   0.9 KB
  price_model.joblib             4185.0 KB
  study.db                        136.0 KB
time: 12.6 ms (started: 2026-09-21 19:37:01 +05:30)


## 5.4 One model layer, two front doors

Both apps in Parts 6 and 7 need the same three things: load the pipelines, encode the
categoricals with the *same* dictionary used at training time, and return a price. Writing that
twice is how the two surfaces drift apart.

So it is written once, in `milepost.py`, sitting next to this notebook:

In [37]:
print(Path("milepost.py").read_text())

"""Milepost's model layer.

Part 5 of the notebook writes two models and a model card into artifacts/.
Everything that serves a prediction -- the Streamlit app, the Gradio demo -- imports
this module and calls model_pred().

A module body runs exactly once per Python process, so the joblib.load() calls below
happen at import time and never again, no matter how many requests (or Streamlit
reruns) follow.
"""

import json
from pathlib import Path

import joblib
import pandas as pd

ARTIFACTS = Path(__file__).parent / "artifacts"

# The same dictionary that encoded the columns at training time. If this drifts
# from the training code, the model keeps predicting -- just wrongly.
encode_dict = {
    "fuel_type": {"Diesel": 1, "Petrol": 2, "CNG": 3, "LPG": 4, "Electric": 5},
    "transmission_type": {"Manual": 1, "Automatic": 2},
    "seller_type": {"Dealer": 1, "Individual": 2, "Trustmark Dealer": 3},
}

FEATURES = [
    "year",
    "seller_type",
    "km_driven",
    "fuel_type",
    "tran

In [38]:
import milepost

print("tuned   :", milepost.model_pred(2018, "Dealer", 45000, "Petrol", "Manual", 18.5, 1200, 85.0, 5), "lakhs")
print("baseline:", milepost.model_pred(2018, "Dealer", 45000, "Petrol", "Manual", 18.5, 1200, 85.0, 5, tuned=False), "lakhs")
print()
print("model card:", json.dumps(milepost.CARD, indent=2)[:400], "...")

tuned   : 5.08 lakhs
baseline: 5.1 lakhs

model card: {
  "model": "XGBRegressor (no preprocessing -- trees are scale-invariant)",
  "target": "selling_price (lakhs)",
  "features": [
    "year",
    "seller_type",
    "km_driven",
    "fuel_type",
    "transmission_type",
    "mileage",
    "engine",
    "max_power",
    "seats"
  ],
  "study_name": "milepost-price",
  "storage": "artifacts/study.db",
  "n_trials": 30,
  "best_trial": 23,
  "best_pa ...
time: 9.57 ms (started: 2026-09-21 19:37:01 +05:30)


The `joblib.load()` calls are in the module body, so they run **once per Python process** no
matter how many predictions follow. That detail is about to matter a great deal.

```
P0 setup  P1 the baseline  P2 optuna  P3 vs grid search  P4 pruning  P5 the artifact  [ P6 STREAMLIT ]  P7 gradio  P8 ship
```

# Part 6 — Streamlit: the surface for a human

The model is tuned, measured and on disk. It is still unusable by anyone who cannot open a
Python REPL.

Streamlit's bargain: **you write a Python script, it becomes a web page.** No HTML, no
JavaScript, no routes, no callbacks. In exchange you accept one unusual rule.

## 6.1 The one rule: the whole script re-runs

There is no event handler. When anyone moves a slider, Streamlit **re-executes your script
from line 1 to the end**, with the widget now returning its new value.

```mermaid
flowchart LR
    A["User drags the<br/>mileage slider"] --> B["Streamlit re-runs<br/>the entire script"]
    B --> C["st.slider(...) now<br/>returns 18.5"]
    C --> D["Every line below it<br/>recomputes"]
    D --> E["Page redraws"]
    E --> A
```

Almost every Streamlit surprise follows from that one rule:

| What you expect | What actually happens |
|---|---|
| `count = count + 1` accumulates | It resets to the starting value on every interaction |
| The model loads when the app starts | `joblib.load()` on line 12 runs on **every** click |
| A variable survives a button press | It does not — unless it is in `st.session_state` |

## 6.2 A tour of the widgets, with no model in sight

Before wiring the model in, here is every widget you will need, on a plain table of daily price
history. `@st.cache_data` tells Streamlit *"the script re-runs, but this result does not
change — reuse it"*, which is the first half of the answer to the re-run rule.

In [39]:
%%writefile ticker_app.py
import datetime

import pandas as pd
import streamlit as st

st.title("Widget tour")

# The script re-runs on every interaction. Without this decorator the CSV would be
# re-read from disk on every single slider drag.
@st.cache_data
def load_history():
    return pd.read_csv("data/ticker_history.csv", index_col="Date", parse_dates=True)

history = load_history()

symbol = st.text_input("Symbol", "AAPL")

col1, col2 = st.columns(2)
with col1:
    start_date = st.date_input("From", datetime.date(2019, 1, 1))
with col2:
    end_date = st.date_input("To", datetime.date(2022, 12, 31))

window = history.loc[str(start_date):str(end_date)]

st.write(f"### {symbol} — {len(window)} trading days")
st.dataframe(window.head(10))

st.write("## Closing price")
st.line_chart(window["Close"])

st.write("## Volume")
st.line_chart(window["Volume"])

col1, col2 = st.columns(2)
with col1:
    st.write("## Open")
    st.line_chart(window["Open"])
with col2:
    st.write("## High")
    st.line_chart(window["High"])

# A slider that feeds a real computation: every drag re-runs the loop below.
st.write("## Exponential moving average")
alpha = st.slider("Alpha", min_value=0.01, max_value=1.0, value=0.10, step=0.01)

ema_values = []
ema = window["Close"].iloc[0]
ema_values.append(ema)
for i in range(1, len(window)):
    ema = alpha * window["Close"].iloc[i] + (1 - alpha) * ema
    ema_values.append(ema)

window = window.assign(EMA=ema_values)
st.line_chart(window[["Close", "EMA"]])

num = st.number_input("Insert a number")
if st.button("Calculate square"):
    st.text(f"Square of {num} is {num ** 2}")

Overwriting ticker_app.py
time: 831 µs (started: 2026-09-21 19:37:01 +05:30)


Run it from a terminal — not from this notebook. A Streamlit process runs until you stop it,
so it needs a terminal you own:

```bash
streamlit run ticker_app.py
```

It opens `http://localhost:8501`. Drag the **Alpha** slider: low alpha gives a smooth lagging
line, alpha near 1 tracks the close almost exactly. That whole EMA loop re-runs on every drag,
which is exactly the re-run rule doing its job.

Stop it with `Ctrl-C` in the same terminal.

### The widgets, as a reference

| Call | Returns | Use for |
|---|---|---|
| `st.title` / `st.write` / `st.text` | — | Headings and prose; `st.write` renders almost anything |
| `st.text_input(label, default)` | `str` | Free text |
| `st.number_input(label, min_value=, max_value=, value=, step=)` | `float` / `int` | Bounded numbers |
| `st.slider(label, min_value=, max_value=, value=, step=)` | number | A number where the *range* matters |
| `st.selectbox(label, options)` | the chosen option | One of a fixed set |
| `st.date_input(label, default)` | `datetime.date` | Dates |
| `st.button(label)` | `bool` — `True` **only on the run triggered by the click** | Firing an action |
| `st.columns(n)` | list of containers | Side-by-side layout, used with `with` |
| `st.tabs([...])` / `st.sidebar` | containers | Grouping |
| `st.dataframe` / `st.line_chart` / `st.bar_chart` | — | Tables and quick charts |
| `st.session_state` | dict-like | The one thing that survives a re-run |

`st.button` deserves the bold text. It is `True` on the re-run caused by the click and `False`
on every re-run after that — so anything you compute inside `if st.button(...)` disappears the
moment the user touches another widget, unless you put it in `st.session_state`.

## 6.3 What a re-run costs when a model is involved

The re-run rule stops being a curiosity the moment `joblib.load()` is in the script. Measure it.

In [40]:
model_kb = (ARTIFACTS / "price_model.joblib").stat().st_size / 1024

t0 = time.perf_counter()
for _ in range(20):
    joblib.load(ARTIFACTS / "price_model.joblib")
reload_ms = (time.perf_counter() - t0) / 20 * 1000

t0 = time.perf_counter()
for _ in range(20):
    milepost.model_pred(2018, "Dealer", 45000, "Petrol", "Manual", 18.5, 1200, 85.0, 5)
predict_ms = (time.perf_counter() - t0) / 20 * 1000

print(f"model on disk         : {model_kb:7.1f} KB")
print(f"loading the artifact  : {reload_ms:7.2f} ms")
print(f"one prediction        : {predict_ms:7.2f} ms")
print(f"ratio                 : {reload_ms / predict_ms:7.1f}x")

model on disk         :  4185.0 KB
loading the artifact  :    4.52 ms
one prediction        :    1.00 ms
ratio                 :     4.5x
time: 111 ms (started: 2026-09-21 19:37:01 +05:30)


Worth reading that honestly rather than as an argument for caching: a reload costs a few
milliseconds — a handful of predictions' worth — and nobody would feel it on a page that
redraws in a tenth of a second.

But look at the file size next to Part 5's other artifact. The tuned model is roughly **ten
times the baseline's size**, because the search settled on hundreds of deeper trees.
Tuning made the artifact an order of magnitude bigger, and the load time went with it while
the prediction stayed flat.

That is the whole caching argument, and it is a *function of model size* rather than a
property of today's numbers. Swap this regressor for an ensemble at 200 MB or a transformer
checkpoint at 2 GB and the ratio goes to hundreds — with every keystroke in every open browser
tab paying it. Cache because the rule has to keep holding when the model grows.

Two ways to do it, and they are not interchangeable:

- **`@st.cache_data`** — for *data*: DataFrames, query results, anything Streamlit can hash and copy. Returns a copy, so mutating it is safe.
- **`@st.cache_resource`** — for *connections and models*: objects that are expensive, unhashable, and meant to be shared. Returns the **same object** to every session.

```python
@st.cache_resource          # right for a model
def get_model():
    return joblib.load("artifacts/price_model.joblib")
```

`milepost.py` sidesteps the choice entirely: its `joblib.load()` calls are in the module body,
and Python caches imported modules per process. The app below imports and never thinks about it.

## 6.4 The real app

This is where the two halves of the notebook meet. The app serves the **tuned** pipeline, the
sidebar reports which study and trial produced it straight from the model card, and a checkbox
turns on a side-by-side comparison against the untuned baseline — which is how the analyst
using this sees that Part 3 was worth doing.

In [41]:
%%writefile price_app.py
import pandas as pd
import streamlit as st

# The module body runs once per process and loads the artifacts. Nothing here
# re-loads them on a re-run.
from milepost import CARD, model_pred

st.set_page_config(page_title="Milepost", layout="centered")
st.title("Milepost — what is this car worth?")

with st.sidebar:
    st.header("Model in use")
    st.metric("Test MAE (lakhs)", CARD["test_mae"], delta=round(CARD["test_mae"] - CARD["baseline_test_mae"], 4), delta_color="inverse")
    st.caption(
        f"study `{CARD['study_name']}` · trial {CARD['best_trial']} of {CARD['n_trials']}"
    )
    st.json(CARD["best_params"], expanded=False)
    compare = st.checkbox("Compare against the untuned baseline")

year = st.slider("Manufacturing year", min_value=1991, max_value=2021, value=2015, step=1)
seller_type = st.selectbox("Seller type", ["Dealer", "Individual", "Trustmark Dealer"])

col1, col2 = st.columns(2)
with col1:
    fuel_type = st.selectbox("Fuel type", ["Diesel", "Petrol", "CNG", "LPG", "Electric"])
    max_power = st.number_input("Max power (bhp)", min_value=5.0, max_value=650.0, value=85.0, step=5.0)
with col2:
    engine = st.number_input("Engine (cc)", min_value=500, max_value=7000, value=1200, step=100)
    km_driven = st.number_input("Kilometres driven", min_value=100, max_value=400000, value=45000, step=5000)

mileage = st.number_input("Mileage (kmpl)", min_value=5.0, max_value=40.0, value=18.5, step=0.5)
transmission_type = st.selectbox("Transmission", ["Manual", "Automatic"])
seats = st.number_input("Seats", min_value=2, max_value=14, value=5, step=1)

car = (year, seller_type, km_driven, fuel_type, transmission_type, mileage, engine, max_power, seats)

# st.button is True only on the re-run the click caused, so the result has to be
# parked in session_state to survive the next widget interaction.
if "quotes" not in st.session_state:
    st.session_state.quotes = []

if st.button("Estimate price", type="primary"):
    price = model_pred(*car)
    st.success(f"Estimated resale price: {price} lakhs")

    if compare:
        untuned = model_pred(*car, tuned=False)
        c1, c2 = st.columns(2)
        c1.metric("Tuned model", f"{price} lakhs")
        c2.metric("Untuned baseline", f"{untuned} lakhs", delta=f"{round(untuned - price, 2)}")

    st.session_state.quotes.append(
        {"year": year, "km": km_driven, "fuel": fuel_type, "price (lakhs)": price}
    )
else:
    st.info("Fill in the details, then press **Estimate price**.")

if st.session_state.quotes:
    st.write("### This session's quotes")
    st.dataframe(pd.DataFrame(st.session_state.quotes), use_container_width=True)

Overwriting price_app.py
time: 1.13 ms (started: 2026-09-21 19:37:01 +05:30)


```bash
streamlit run price_app.py
```

Things to try, because each one demonstrates a rule above:

1. Press **Estimate price**, then drag the year slider. The green box disappears — `st.button` went back to `False` — but the quotes table survives, because it lives in `st.session_state`.
2. Tick **Compare against the untuned baseline** and price a few cars. The two numbers differ by more on unusual cars than on ordinary ones, which is roughly where the tuning bought its MAE.
3. Set kilometres to 400,000 and the year to 2021. The model will still answer. It has never seen that combination and has no way to tell you so.

## 6.5 Where Streamlit runs out

The app works, and for an analyst on your team it is the right answer. Then the partner bank
asks to integrate, and every one of these is a wall:

- There is **no URL that returns a price.** `http://localhost:8501` returns a JavaScript application, not `{"price": 4.87}`.
- There is **no contract.** Nothing declares what fields exist or what types they take.
- There is **no way to call it from code.** Streamlit assumes a browser with a websocket, not a `requests.post`.
- It is **stateful per user.** Each browser tab gets its own Python session — fine for ten analysts, not a model for a service under load.

Streamlit is excellent at what it does and structurally unable to do this. Serving a *program*
rather than a person is a different tool and the subject of the next class.

```
P0 setup  P1 the baseline  P2 optuna  P3 vs grid search  P4 pruning  P5 the artifact  P6 streamlit  [ P7 GRADIO ]  P8 ship
```

# Part 7 — Gradio: the surface for a link

Gradio's bargain is even shorter: **give it a function, and it builds the UI from the
function's inputs and outputs.**

```python
gr.Interface(fn=model_pred, inputs=[...], outputs="text")
```

There is no script re-run rule to learn and no routes to declare. In exchange you give up
layout control — Gradio decides what the page looks like.

The thing Gradio has that Streamlit does not is **`share=True`**: one argument, and it tunnels
your local app to a public `https://....gradio.live` URL that works on anyone's phone for 72
hours. No deployment, no cloud account. That is its real job.

In [42]:
%%writefile gradio_app.py
import gradio as gr

from milepost import CARD, model_pred


def price_with_comparison(*car):
    """Gradio wires one function to one click; this returns both numbers at once."""
    tuned = model_pred(*car)
    untuned = model_pred(*car, tuned=False)
    return f"{tuned} lakhs", f"{untuned} lakhs (untuned baseline)"


with gr.Blocks(title="Milepost") as demo:
    gr.Markdown(
        f"# Milepost\nResale pricing for used cars.\n\n"
        f"Model: study `{CARD['study_name']}`, trial {CARD['best_trial']} of "
        f"{CARD['n_trials']} — test MAE {CARD['test_mae']} lakhs."
    )

    with gr.Row():
        year = gr.Slider(1991, 2021, value=2015, step=1, label="Manufacturing year")
        seller_type = gr.Dropdown(["Dealer", "Individual", "Trustmark Dealer"], value="Dealer", label="Seller type")
    with gr.Row():
        fuel_type = gr.Dropdown(["Diesel", "Petrol", "CNG", "LPG", "Electric"], value="Petrol", label="Fuel type")
        transmission_type = gr.Dropdown(["Manual", "Automatic"], value="Manual", label="Transmission")
    with gr.Row():
        km_driven = gr.Number(value=45000, label="Kilometres driven")
        mileage = gr.Number(value=18.5, label="Mileage (kmpl)")
    with gr.Row():
        engine = gr.Number(value=1200, label="Engine (cc)")
        max_power = gr.Number(value=85.0, label="Max power (bhp)")
        seats = gr.Number(value=5, label="Seats")

    with gr.Row():
        price_out = gr.Textbox(label="Estimated resale price")
        baseline_out = gr.Textbox(label="Before tuning")

    gr.Button("Estimate price", variant="primary").click(
        fn=price_with_comparison,
        inputs=[year, seller_type, km_driven, fuel_type, transmission_type, mileage, engine, max_power, seats],
        outputs=[price_out, baseline_out],
    )

if __name__ == "__main__":
    # share=True also publishes a temporary public https://....gradio.live URL
    demo.launch()

Overwriting gradio_app.py
time: 820 µs (started: 2026-09-21 19:37:01 +05:30)


The `.click(fn=..., inputs=[...], outputs=[...])` line is the whole wiring model: when this
control fires, call that function with the current value of those components, and put the
returned values into those outputs, in order. `model_pred` is used completely unmodified —
Gradio needs nothing from it.

Note the `if __name__ == "__main__":` guard around `demo.launch()`. Without it, merely importing
the module would start a server. With it, the file can be inspected safely and only *running*
it launches anything.

In [43]:
import gradio_app

demo = gradio_app.demo
print("blocks built:", type(demo).__name__)
print("components  :", len(demo.blocks))
print("wired events:", len(demo.fns))
print("\nNo server started — importing only constructed the object.")

blocks built: Blocks
components  : 23
wired events: 1

No server started — importing only constructed the object.
time: 1.46 s (started: 2026-09-21 19:37:01 +05:30)


In [44]:
# The wired function, called directly. This is exactly what a click does.
print(gradio_app.price_with_comparison(2018, "Dealer", 45000, "Petrol", "Manual", 18.5, 1200, 85.0, 5))

('5.08 lakhs', '5.1 lakhs (untuned baseline)')
time: 3.04 ms (started: 2026-09-21 19:37:03 +05:30)


Run it from a terminal:

```bash
python gradio_app.py
```

It serves on `http://127.0.0.1:7860`. To hand it to someone who is not on your network, change
the last line to `demo.launch(share=True)` and re-run — Gradio prints a second, public URL.

Two things to understand about that link before you paste it into a work chat:

- Your laptop is still serving every request. Close the terminal and the link dies.
- It is **public and unauthenticated** for as long as it lives. Anyone with the URL can query your model. Use `demo.launch(share=True, auth=("user", "password"))` if the data is not trivial.

`Ctrl-C` stops it.

## 7.1 Gradio or Streamlit

They overlap enough to argue about and differ enough that the choice is usually obvious:

| | **Streamlit** | **Gradio** |
|---|---|---|
| Mental model | A script that re-runs top to bottom | Functions wired to components |
| You write | The page | The function signature |
| Layout control | Substantial — columns, tabs, sidebar, containers | Limited by design |
| Multi-step app state | `st.session_state`, genuinely usable | Awkward past a couple of fields |
| Instant public link | No — needs hosting | **Yes — `share=True`** |
| Built-in API for the same app | No | Yes — every Gradio app exposes `/gradio_api` |
| Best at | An internal tool someone uses weekly | A demo someone opens once |

The deciding question is lifespan. Something a colleague will use every Tuesday for a year is
Streamlit. Something you need three people to look at before Friday is Gradio.

Neither of them is an API, which is the wall §6.5 ran into and the subject of the next class.

```
P0 setup  P1 the baseline  P2 optuna  P3 vs grid search  P4 pruning  P5 the artifact  P6 streamlit  P7 gradio  [ P8 SHIP ]
```

# Part 8 — What is on disk, and what is next

## 8.1 Everything this notebook produced

In [45]:
apps = sorted(p for p in Path(".").glob("*.py") if not p.name.startswith("_"))
for p in apps + sorted(ARTIFACTS.glob("*")):
    print(f"  {str(p):32s} {p.stat().st_size / 1024:8.1f} KB")

  gradio_app.py                         1.9 KB
  milepost.py                           2.8 KB
  price_app.py                          2.6 KB
  ticker_app.py                         1.6 KB
  artifacts/baseline_model.joblib     392.9 KB
  artifacts/model_card.json             0.9 KB
  artifacts/price_model.joblib       4185.0 KB
  artifacts/study.db                  136.0 KB
time: 806 µs (started: 2026-09-21 19:37:03 +05:30)


In [46]:
%%writefile requirements.txt
optuna==4.9.0
streamlit==1.41.1
gradio==6.27.0
scikit-learn==1.7.2
xgboost==3.2.0
torch==2.10.0
plotly==5.24.1
pandas==2.3.3
joblib==1.4.0

Overwriting requirements.txt
time: 724 µs (started: 2026-09-21 19:37:03 +05:30)


Pinned, not `>=`. An unpinned `scikit-learn` is how a pickle that loaded fine in March stops
loading in June — the artifact carries no code, only state, and the class it expects to be
poured back into has to still exist with the same shape.

## 8.2 The tuning half, in five lines

```python
def objective(trial):
    params = {"max_depth": trial.suggest_int("max_depth", 3, 10), ...}
    return cv_mae(XGBRegressor(**params))          # ONE number, the thing to minimise

study = optuna.create_study(direction="minimize", storage="sqlite:///study.db")
study.optimize(objective, n_trials=60)
```

Everything else in Parts 2–5 is a consequence:

| If you need | Add |
|---|---|
| A search space with `if` in it | Nothing — it is already a function |
| To stop bad trials early | `trial.report(v, step)` + `if trial.should_prune(): raise optuna.TrialPruned()` |
| To survive a kernel restart | `storage=` and `load_if_exists=True` |
| To know which knob mattered | `plot_param_importances(study)` |
| To resume tomorrow | `optuna.load_study(study_name=..., storage=...)` |

And the rule that comes before all of it, from Part 1: **check that your metric can tell two
models apart before you spend an hour optimising it.**

## 8.3 The decision, in one rule

```mermaid
flowchart TD
    Q{"Who opens it?"}
    Q -->|"A person, in a browser"| H{"How long does it live?"}
    Q -->|"Another program"| N["Not in this notebook<br/>— that is Flask and FastAPI,<br/>next class"]
    H -->|"An internal tool, used for months"| S["Streamlit<br/>layout, session state, a script that re-runs"]
    H -->|"A link, shown once this week"| G["Gradio<br/>function in, UI out; share=True"]
```

## 8.4 What the next class picks up

§6.5 listed four things Streamlit structurally cannot do, and every one of them is the same
missing thing: **a URL that returns `{"price": 4.87}`**. The next class builds it twice —
once with **Flask**, to see what a route actually is, and once with **FastAPI**, to see what
declaring the contract buys you.

`milepost.py` is already the seam. Both of those frameworks will import the same
`model_pred` this notebook's two apps import, without changing a line of it — which is the
real reason the model layer was pulled out of the app in the first place.